In [ ]:
!pip install "trl>=0.24.0,<0.25" "peft>=0.17.0,<0.18" --force-reinstall

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch
import bitsandbytes as bnb

# ----------------------------
# 配置：使用本地模型路径
# ----------------------------
local_model_path = "/mnt/workspace/.cache/modelscope/models/qwen/Qwen2-1.5B-Instruct"  # 👈 替换为你的实际路径
output_dir = "./qwen2-1.5b-en2zh-qlora-test"
max_seq_length = 512
num_samples = 1000

# ----------------------------
# 加载 tokenizer（从本地）
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    use_fast=False  # Qwen 建议关闭 fast tokenizer
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# 加载 4-bit 模型（从本地）
# ----------------------------
print("Loading model from local path in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    load_in_4bit=True,
    device_map="auto",
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading model from local path in 4-bit...


In [3]:
# ----------------------------
# LoRA 配置
# ----------------------------
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split(".")
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    return list(lora_module_names)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)


model = get_peft_model(model, lora_config)

model.print_trainable_parameters()  # 应该显示 ~10M 可训练参数

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [5]:
# ----------------------------
# 加载前 1000 条 WikiMatrix 数据
# ----------------------------
def load_moses_data(en_file, zh_file, n=1000):
    en_lines, zh_lines = [], []
    with open(en_file, encoding="utf-8") as f_en, open(zh_file, encoding="utf-8") as f_zh:
        for i, (en_line, zh_line) in enumerate(zip(f_en, f_zh)):
            if i >= n:
                break
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            if en_line and zh_line:
                en_lines.append(en_line)
                zh_lines.append(zh_line)
    print(f"Loaded {len(en_lines)} sentence pairs.")
    return Dataset.from_dict({"en": en_lines, "zh": zh_lines})

path_prefix = '/mnt/workspace/modelscope/data/'
dataset = load_moses_data(path_prefix+"WikiMatrix.en-zh.en", path_prefix+"WikiMatrix.en-zh.zh", num_samples)


Loaded 1000 sentence pairs.


In [6]:
print("Dataset format:", dataset.format)

Dataset format: {'type': None, 'format_kwargs': {}, 'columns': ['en', 'zh'], 'output_all_columns': False}


In [7]:
for i, example in enumerate(dataset):
    print(f"--- Sample {i} ---")
    print("en:", repr(example["en"]))
    print("zh:", repr(example["zh"]))
    if i >= 5:  # 只打印前 6 行，避免刷屏
        break

--- Sample 0 ---
en: 'This is an alleviation from your Lord and a mercy.'
zh: '這是你們的主所降示的減輕和慈恩。'
--- Sample 1 ---
en: 'The Supreme Lord of the indescribable grace (Alekha) is worshiped.'
zh: '以昭事上主，闡揚天主聖教為本。'
--- Sample 2 ---
en: 'بِأَنَّ رَبَّكَ أَوْحَى لَهَا Because your Lord has commanded it.'
zh: '你的主的言辞，诚实极了，公平极了。'
--- Sample 3 ---
en: "The Lord's Prayer: The Beatitudes."
zh: '祈求圣主明鉴施恩。'
--- Sample 4 ---
en: 'Blessed is he who comes in the name of the Lord!'
zh: '因上主之名而來的，當受讚頌!'
--- Sample 5 ---
en: 'For example, ன is ṉa (with the inherent a) and ன் is ṉ (without a vowel).'
zh: '例如ன是ṉa（有母音a），而ன்是ṉ（沒有母音）。'


In [8]:
def formatting_func(example):
    if not isinstance(example, dict):
        raise TypeError(f"Expected dict, got {type(example)}: {example}")
    if "en" not in example:
        raise KeyError(f"'en' not in example. Keys: {list(example.keys())}. Example: {example}")
    if "zh" not in example:
        raise KeyError(f"'zh' not in example. Keys: {list(example.keys())}")
    
    prompt = (
        "You are a professional translator.\n"
        "<|im_start|>user\n"
        f"Translate the following English text to Chinese:\n{example['en']}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['zh']}<|im_end|>"
    )
    inputs = tokenizer(prompt, truncation=True, max_length=256, add_special_tokens=True)
    truncated = tokenizer.decode(inputs["input_ids"], skip_special_tokens=False)
    return truncated


In [9]:
# --- 训练参数 ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="constant",
    report_to="none",
)

In [10]:
# --- Trainer（仅传支持的参数）---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,  # 注意：不是 tokenizer=
    # 不传 packing / max_seq_length / data_collator 等
)

Truncating train dataset: 100%|██████████| 1000/1000 [00:00<00:00, 371637.78 examples/s]
df: /root/.triton/autotune: 没有那个文件或目录


In [11]:
# --- 开始训练 ---
print("🚀 Starting training...")
trainer.train()
print("✅ Done!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


🚀 Starting training...


/usr/local/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.260800
20,1.656800
30,1.591800
40,1.471400
50,1.547200
60,1.576200
70,1.545000
80,1.487200
90,1.450800
100,1.526100


✅ Done!


In [12]:
trainer.save_model()